<a href="https://colab.research.google.com/github/PromyotKatarat/14_agent_engineering/blob/main/14_agent_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. ดึงโปรเจกต์มาทั้งหมดก่อน
!git clone https://github.com/rohitg00/ai-engineering-from-scratch.git

# 2. ย้ายพาธเข้าไปที่โฟลเดอร์หลักของ Agent Engineering
%cd ai-engineering-from-scratch/phases/14-agent-engineering/

# 3. ลองเช็กดูซิว่ามีโฟลเดอร์บทเรียนอะไรให้เล่นบ้าง
!ls -l

Cloning into 'ai-engineering-from-scratch'...
remote: Enumerating objects: 15819, done.
remote: Counting objects: 100% (7444/7444), done.
remote: Compressing objects: 100% (3245/3245), done.
remote: Total 15819 (delta 4564), reused 4206 (delta 4199), pack-reused 8375 (from 1)
Receiving objects: 100% (15819/15819), 8.79 MiB | 13.91 MiB/s, done.
Resolving deltas: 100% (7267/7267), done.
/content/ai-engineering-from-scratch/phases/14-agent-engineering
total 172
drwxr-xr-x 7 root root 4096 Jun  2 07:58 01-the-agent-loop
drwxr-xr-x 7 root root 4096 Jun  2 07:58 02-rewoo-plan-and-execute
drwxr-xr-x 7 root root 4096 Jun  2 07:58 03-reflexion-verbal-rl
drwxr-xr-x 7 root root 4096 Jun  2 07:58 04-tree-of-thoughts-lats
drwxr-xr-x 7 root root 4096 Jun  2 07:58 05-self-refine-and-critic
drwxr-xr-x 7 root root 4096 Jun  2 07:58 06-tool-use-and-function-calling
drwxr-xr-x 7 root root 4096 Jun  2 07:58 07-memory-virtual-context-memgpt
drwxr-xr-x 7 root root 4096 Jun  2 07:58 08-memory-blocks-sleep-ti

In [2]:
!python 01-the-agent-loop/code/main.py

TOY REACT LOOP — Phase 14, Lesson 01

[00    user] What is 120 plus 15% tax, stored in kv?
[01 thought] store the base price
[02  action] kv_set({'key': 'base', 'value': '120'}) -> stored base
[03 thought] compute 15% tax
[04  action] calculator({'expr': '120 * 0.15'}) -> 18.0
[05 thought] store the tax
[06  action] kv_set({'key': 'tax', 'value': '18.0'}) -> stored tax
[07 thought] compute total
[08  action] calculator({'expr': '120 + 18.0'}) -> 138.0
[09 thought] confirm stored values
[10  action] kv_get({'key': 'base'}) -> 120
[11   final] the total including 15% tax is 138.0

final answer: the total including 15% tax is 138.0
turns used:   5
tools used:   ['calculator', 'kv_get', 'kv_set']


In [8]:
# %load 01-the-agent-loop/code/main.py
"""Toy ReAct agent loop — stdlib only.

Implements the five ingredients from docs/en.md:
  1. message buffer
  2. tool registry
  3. stop condition
  4. turn budget
  5. observation formatter

ToyLLM is a scripted policy so the loop runs offline and deterministic. Swap
ToyLLM for a real provider client and the control flow is identical.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable


@dataclass
class ToolCall:
    name: str
    args: dict[str, Any]


@dataclass
class Turn:
    kind: str
    content: str
    tool_call: ToolCall | None = None
    observation: str | None = None


class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., str]] = {}

    def register(self, name: str, fn: Callable[..., str]) -> None:
        self._tools[name] = fn

    def names(self) -> list[str]:
        return sorted(self._tools)

    def dispatch(self, call: ToolCall) -> str:
        fn = self._tools.get(call.name)
        if fn is None:
            return f"error: unknown tool {call.name!r}"
        try:
            return fn(**call.args)
        except TypeError as e:
            return f"error: bad args for {call.name}: {e}"
        except Exception as e:
            return f"error: {type(e).__name__}: {e}"


def calculator(expr: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not set(expr).issubset(allowed):
        return "error: illegal character in expr"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {type(e).__name__}: {e}"


class KVStore:
    def __init__(self) -> None:
        self._store: dict[str, str] = {}

    def get(self, key: str) -> str:
        return self._store.get(key, f"missing:{key}")

    def set(self, key: str, value: str) -> str:
        self._store[key] = value
        return f"stored {key}"


class ToyLLM:
    """Scripted ReAct policy. Returns one assistant turn per call.

    Each script entry is either ('thought', text) plus ('action', name, args)
    or ('finish', text). The loop runs through the script in order.
    """

    def __init__(self, script: list[dict[str, Any]]) -> None:
        self.script = script
        self.cursor = 0

    def respond(self, history: list[Turn]) -> dict[str, Any]:
        if self.cursor >= len(self.script):
            return {"kind": "finish", "content": "no more actions"}
        entry = self.script[self.cursor]
        self.cursor += 1
        return entry


@dataclass
class AgentLoop:
    llm: ToyLLM
    tools: ToolRegistry
    max_turns: int = 12
    max_tool_calls_per_turn: int = 2  # 🌟 จุดเพิ่มที่ 1: กำหนดโควตาใช้เครื่องมือสูงสุดต่อรอบ (Turn)
    history: list[Turn] = field(default_factory=list)

    def run(self, user_message: str) -> str:
        self.history.append(Turn(kind="user", content=user_message))

        for step in range(self.max_turns):
            reply = self.llm.respond(self.history)

            # 1. เงื่อนไขจบงานแบบเดิม (Explicit Finish)
            if reply["kind"] == "finish":
                self.history.append(Turn(kind="final", content=reply["content"]))
                return reply["content"]

            # --- 🌟 จุดเพิ่มโจทย์ข้อ 2: ดักจับทางลัด no_tool_calls -> done ---
            # ถ้าโมเดลส่งผลลัพธ์มา แต่ "ไม่มี" การระบุชื่อ action หรือ action เป็นค่าว่าง/None
            if "action" not in reply or reply["action"] is None:
                final_content = reply.get("thought", "Done (No tools called)")
                self.history.append(Turn(kind="final", content=f"[Auto-Done]: {final_content}"))
                return final_content
            # -------------------------------------------------------------

            # (โค้ดส่วนคิดและรันเครื่องมือเดิม...)
            thought = reply.get("thought", "")
            self.history.append(Turn(kind="thought", content=thought))
            call = ToolCall(name=reply["action"], args=reply.get("args", {}))
            observation = self.tools.dispatch(call)
            self.history.append(Turn(kind="action", content=call.name, tool_call=call, observation=observation))

        self.history.append(Turn(kind="final", content="budget exhausted"))
        return "budget exhausted"


def pretty_trace(history: list[Turn]) -> None:
    for i, turn in enumerate(history):
        tag = f"[{i:02d} {turn.kind:>7}]"
        if turn.kind == "user":
            print(f"{tag} {turn.content}")
        elif turn.kind == "thought":
            print(f"{tag} {turn.content}")
        elif turn.kind == "action":
            call = turn.tool_call
            assert call is not None
            print(f"{tag} {call.name}({call.args}) -> {turn.observation}")
        elif turn.kind == "final":
            print(f"{tag} {turn.content}")


def build_demo_agent() -> AgentLoop:
    tools = ToolRegistry()
    tools.register("calculator", calculator)
    kv = KVStore()
    tools.register("kv_get", kv.get)
    tools.register("kv_set", kv.set)

    script: list[dict[str, Any]] = [
        {"kind": "action", "thought": "store the base price",
         "action": "kv_set", "args": {"key": "base", "value": "120"}},
        {"kind": "action", "thought": "compute 15% tax",
         "action": "calculator", "args": {"expr": "120 * 0.15"}},
        {"kind": "action", "thought": "store the tax",
         "action": "kv_set", "args": {"key": "tax", "value": "18.0"}},
        {"kind": "action", "thought": "compute total",
         "action": "calculator", "args": {"expr": "120 + 18.0"}},
        {"kind": "action", "thought": "confirm stored values",
         "action": "kv_get", "args": {"key": "base"}},
        {"kind": "finish", "content": "the total including 15% tax is 138.0"},
    ]
    return AgentLoop(llm=ToyLLM(script), tools=tools, max_turns=10)


def main() -> None:
    print("=" * 70)
    print("TOY REACT LOOP — Phase 14, Lesson 01")
    print("=" * 70)
    agent = build_demo_agent()
    final = agent.run("What is 120 plus 15% tax, stored in kv?")
    print()
    pretty_trace(agent.history)
    print()
    print(f"final answer: {final}")
    print(f"turns used:   {len([t for t in agent.history if t.kind == 'action'])}")
    print(f"tools used:   {agent.tools.names()}")


if __name__ == "__main__":
    main()


TOY REACT LOOP — Phase 14, Lesson 01

[00    user] What is 120 plus 15% tax, stored in kv?
[01 thought] store the base price
[02  action] kv_set({'key': 'base', 'value': '120'}) -> stored base
[03 thought] compute 15% tax
[04  action] calculator({'expr': '120 * 0.15'}) -> 18.0
[05 thought] store the tax
[06  action] kv_set({'key': 'tax', 'value': '18.0'}) -> stored tax
[07 thought] compute total
[08  action] calculator({'expr': '120 + 18.0'}) -> 138.0
[09 thought] confirm stored values
[10  action] kv_get({'key': 'base'}) -> 120
[11   final] the total including 15% tax is 138.0

final answer: the total including 15% tax is 138.0
turns used:   5
tools used:   ['calculator', 'kv_get', 'kv_set']
